# Финальный ретро-прогон кассовой потребности (с расписанием)

Копия итогового adaptive-алгоритма с уточнённой логикой закрытых дней:

- рабочий день = открыта хотя бы по одному из графиков `wtimecorp` / `wtimepriv`;
- праздники (`HOL_MSK_FLG = 1`) всегда закрыты;
- открытый день без операций → факт `0` (не «касса закрыта»);
- для сравнения со старым прогнозом на открытых днях с `0`/пропуском берётся предыдущее ненулевое значение старой модели.

Исходный ноутбук `cashdesk_final_adaptive_retro.ipynb` не меняется.


In [ ]:
import os
import re
import time
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, FrozenSet, List, Optional, Set, Tuple

import numpy as np
import pandas as pd
from joblib import Parallel, delayed, parallel_backend
from prefect.blocks.system import Secret
from sqlalchemy import text
from statsmodels.tsa.statespace.sarimax import SARIMAX
from toolbox import oracle

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 200)


In [ ]:
SOURCE_TABLE = "AIDA2.AIDA_TRS_DTM_CASHOP@aida"
OLD_FORECAST_TABLE = "AIDA2.AIDA_TRS_DTM_FORECAST_HISTORY@aida"
ORACLE_TARGET_TABLE = "EMA_CASHDESK_PREDS_2M"

RETRO_DATE_FROM = pd.Timestamp("2026-04-15")
RETRO_DATE_TO = pd.Timestamp("2026-06-15")
HISTORY_MONTHS = 12
ACTIVE_LOOKBACK_MONTHS = 1
DATA_LAG_DAYS = 2
FORECAST_DAYS = 30
OLD_LOOKBACK_DAYS = 30
MODEL_STEPS = FORECAST_DAYS + DATA_LAG_DAYS - 1

ERROR_WINDOW_GRID = [56, 84, 112, 168, 224, 280, 365]
SHRINKAGE_GRID = [5.0, 10.0, 20.0, 40.0, 80.0]
PRETEST_SELECTION_DAYS = 30
HISTORICAL_ERROR_STEP_DAYS = 7
MIN_SCALE_ROWS = 5
MIN_SARIMA_OBS = 90

INITIAL_QUANTILE = 0.15
TARGET_BREACH_RATE = 0.15
ADAPTIVE_GAMMA = 0.20
ADAPTIVE_QUANTILE_MIN = 0.01
ADAPTIVE_QUANTILE_MAX = 0.25
ADAPTIVE_MIN_UPDATE_ROWS = 30

SARIMA_ORDER = (1, 1, 1)
SARIMA_SEASONAL_ORDER = (1, 0, 1, 7)
CASH_NEED_CLIP_UPPER = 0.0
FORECAST_SCALE_MULTIPLIER = 20.0
EXCLUDE_DATE_RANGES = [("2025-09-01", "2025-11-01")]

PRETEST_REPORT_DATE = RETRO_DATE_FROM - pd.Timedelta(days=DATA_LAG_DAYS)
PRETEST_SELECTION_DATE_TO = PRETEST_REPORT_DATE
PRETEST_SELECTION_DATE_FROM = (
    PRETEST_SELECTION_DATE_TO
    - pd.Timedelta(days=PRETEST_SELECTION_DAYS - 1)
)
pretest_selection_dates = pd.date_range(
    PRETEST_SELECTION_DATE_FROM,
    PRETEST_SELECTION_DATE_TO,
    freq="D",
)

MAX_ERROR_WINDOW_DAYS = max(ERROR_WINDOW_GRID)
EARLIEST_SELECTION_REPORT_DATE = (
    PRETEST_SELECTION_DATE_FROM - pd.Timedelta(days=DATA_LAG_DAYS)
)
EARLIEST_ERROR_FACT_DATE = (
    EARLIEST_SELECTION_REPORT_DATE
    - pd.Timedelta(days=MAX_ERROR_WINDOW_DAYS - 1)
)
EARLIEST_HISTORICAL_SCORE_DATE = (
    EARLIEST_ERROR_FACT_DATE
    - pd.Timedelta(days=FORECAST_DAYS - 1)
)
LATEST_HISTORICAL_SCORE_DATE = (
    PRETEST_REPORT_DATE - pd.Timedelta(days=FORECAST_DAYS - 1)
)
historical_error_score_dates = pd.date_range(
    EARLIEST_HISTORICAL_SCORE_DATE,
    LATEST_HISTORICAL_SCORE_DATE,
    freq="{}D".format(HISTORICAL_ERROR_STEP_DAYS),
)
calibration_score_dates = pd.DatetimeIndex(sorted(set(
    historical_error_score_dates.tolist()
    + pretest_selection_dates.tolist()
)))

EARLIEST_HISTORICAL_REPORT_DATE = (
    EARLIEST_HISTORICAL_SCORE_DATE
    - pd.Timedelta(days=DATA_LAG_DAYS)
)
DATA_DATE_FROM = (
    EARLIEST_HISTORICAL_REPORT_DATE
    - pd.DateOffset(months=HISTORY_MONTHS)
    + pd.Timedelta(days=1)
)
score_dates = pd.date_range(RETRO_DATE_FROM, RETRO_DATE_TO, freq="D")
FACT_DATE_TO_EXCLUSIVE = score_dates.max() + pd.Timedelta(days=FORECAST_DAYS)
OLD_DATE_FROM = RETRO_DATE_FROM - pd.Timedelta(days=OLD_LOOKBACK_DAYS)
OLD_DATE_TO_EXCLUSIVE = RETRO_DATE_TO + pd.Timedelta(days=1)

N_JOBS = 16
LOAD_RAW_FROM_CACHE = False
USE_CHECKPOINTS = True
RAW_CACHE_PATH = Path("data/raw/cashdesk_final_adaptive_schedule_raw.parquet")
OLD_CACHE_PATH = Path("data/raw/cashdesk_final_adaptive_schedule_old.parquet")
SCHEDULE_CACHE_PATH = Path(
    "data/raw/cashdesk_final_adaptive_schedule_hours.parquet"
)
HOLIDAY_CACHE_PATH = Path(
    "data/raw/cashdesk_final_adaptive_schedule_holidays.parquet"
)
OUTPUT_DIR = Path("outputs/final_adaptive_retro_schedule")
CALIBRATION_CHECKPOINT_DIR = OUTPUT_DIR / "calibration_checkpoints"
RETRO_CHECKPOINT_DIR = OUTPUT_DIR / "retro_checkpoints"
REPORT_DIR = OUTPUT_DIR / "reports"

print("Ретро: {} — {} ({} даты)".format(
    RETRO_DATE_FROM.date(), RETRO_DATE_TO.date(), len(score_dates)
))
print("Предтестовый выбор окна: {} — {} ({} дат)".format(
    PRETEST_SELECTION_DATE_FROM.date(),
    PRETEST_SELECTION_DATE_TO.date(),
    len(pretest_selection_dates),
))
print("Исходные данные: {} — {}".format(
    DATA_DATE_FROM.date(),
    (FACT_DATE_TO_EXCLUSIVE - pd.Timedelta(days=1)).date(),
))
print("Исторические full-12m точки: {}".format(
    len(historical_error_score_dates)
))


In [ ]:
USERNAME_CDW = "sb_analytics"
engine_cdw = None


async def create_cdw_engine():
    password_cdw = (await Secret.load("pass-sb-analytics")).get()
    return oracle.create_engine_cdw(USERNAME_CDW, password_cdw)


SCHEDULE_QUERY = """
select
    e.codefem,
    e.c_name,
    e.codeibsoretail,
    rtl_dep.c_code_dblink,
    tp.wtimecorp,
    tp.wtimepriv
from ema_working_cashdesks e
left join Ods.ODS_PRX_BANKOFFICE tp
    on tp.CODEFEM = e.codefem
   and tp.dml_type_cd <> 'D'
   and tp.statetp = 'действует'
left join ods.ods_rtl_depart rtl_dep
    on rtl_dep.c_code = tp.codeibsoretail
   and rtl_dep.dml_type_cd <> 'D'
order by e.codefem
"""


if LOAD_RAW_FROM_CACHE:
    raw_df = pd.read_parquet(RAW_CACHE_PATH)
    old_history_df = pd.read_parquet(OLD_CACHE_PATH)
    schedule_df = pd.read_parquet(SCHEDULE_CACHE_PATH)
    holiday_df = pd.read_parquet(HOLIDAY_CACHE_PATH)
else:
    engine_cdw = await create_cdw_engine()
    source_query = """
    select
        atdtmco_cashdesk_name,
        atdtmco_cashdesk_name_trn,
        atdtmco_cashdesk_code,
        atdtmco_calday,
        atdtmco_saldo_turn,
        atdtmco_ns
    from {source_table}
    where atdtmco_calday >= date '{date_from}'
      and atdtmco_calday < date '{date_to}'
    """.format(
        source_table=SOURCE_TABLE,
        date_from=DATA_DATE_FROM.date().isoformat(),
        date_to=FACT_DATE_TO_EXCLUSIVE.date().isoformat(),
    )
    for exclude_start, exclude_end in EXCLUDE_DATE_RANGES:
        source_query += (
            "\n  and not (atdtmco_calday >= date '{}' "
            "and atdtmco_calday < date '{}')"
        ).format(exclude_start, exclude_end)

    old_query = """
    select cashdesk_name, calday, flow_minimum, forecast_model, forecast_time
    from {old_table}
    where calday >= date '{date_from}'
      and calday < date '{date_to}'
    """.format(
        old_table=OLD_FORECAST_TABLE,
        date_from=OLD_DATE_FROM.date().isoformat(),
        date_to=OLD_DATE_TO_EXCLUSIVE.date().isoformat(),
    )
    holiday_query = """
    select cld_day_dt, hol_msk_flg
    from ORL.ORL_CLD_PROD_CALENDAR
    where cld_day_dt >= date '{date_from}'
      and cld_day_dt < date '{date_to}'
    """.format(
        date_from=DATA_DATE_FROM.date().isoformat(),
        date_to=FACT_DATE_TO_EXCLUSIVE.date().isoformat(),
    )
    with engine_cdw.connect() as conn:
        raw_df = pd.read_sql(text(source_query), conn)
        old_history_df = pd.read_sql(text(old_query), conn)
        schedule_df = pd.read_sql(text(SCHEDULE_QUERY), conn)
        holiday_df = pd.read_sql(text(holiday_query), conn)

    RAW_CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)
    raw_df.to_parquet(RAW_CACHE_PATH, index=False)
    old_history_df.to_parquet(OLD_CACHE_PATH, index=False)
    schedule_df.to_parquet(SCHEDULE_CACHE_PATH, index=False)
    holiday_df.to_parquet(HOLIDAY_CACHE_PATH, index=False)

raw_df.columns = raw_df.columns.str.lower()
old_history_df.columns = old_history_df.columns.str.lower()
schedule_df.columns = schedule_df.columns.str.lower()
holiday_df.columns = holiday_df.columns.str.lower()

raw_df["atdtmco_calday"] = pd.to_datetime(
    raw_df["atdtmco_calday"]
).dt.normalize()
old_history_df["calday"] = pd.to_datetime(
    old_history_df["calday"]
).dt.normalize()
old_history_df["forecast_time"] = pd.to_datetime(
    old_history_df["forecast_time"], errors="coerce"
)
holiday_df["cld_day_dt"] = pd.to_datetime(
    holiday_df["cld_day_dt"]
).dt.normalize()
holiday_df["hol_msk_flg"] = pd.to_numeric(
    holiday_df["hol_msk_flg"], errors="coerce"
).fillna(0).astype(int)

raw_df["atdtmco_cashdesk_code"] = (
    raw_df["atdtmco_cashdesk_code"].astype(str).str.strip()
)
schedule_df["c_code_dblink"] = (
    schedule_df["c_code_dblink"].astype(str).str.strip()
)
schedule_df.loc[
    schedule_df["c_code_dblink"].isin(["", "None", "nan", "NaN"]),
    "c_code_dblink",
] = pd.NA

daily_df = (
    raw_df
    .sort_values(["atdtmco_cashdesk_name", "atdtmco_calday"])
    .groupby(
        ["atdtmco_cashdesk_name", "atdtmco_calday"],
        as_index=False,
        dropna=False,
    )
    .agg(
        atdtmco_cashdesk_name_trn=(
            "atdtmco_cashdesk_name_trn",
            "last",
        ),
        atdtmco_cashdesk_code=(
            "atdtmco_cashdesk_code",
            "last",
        ),
        atdtmco_saldo_turn_fact=(
            "atdtmco_saldo_turn",
            lambda values: values.sum(min_count=1),
        ),
        atdtmco_ns_daily_min_raw=("atdtmco_ns", "min"),
    )
    .rename(columns={"atdtmco_calday": "calday"})
    .sort_values(["atdtmco_cashdesk_name", "calday"])
    .reset_index(drop=True)
)
daily_df["atdtmco_ns_fact"] = pd.to_numeric(
    daily_df["atdtmco_ns_daily_min_raw"], errors="coerce"
).clip(upper=CASH_NEED_CLIP_UPPER)

old_history_dedup_df = (
    old_history_df
    .sort_values("forecast_time")
    .drop_duplicates(["cashdesk_name", "calday"], keep="last")
    .reset_index(drop=True)
)
old_history_dedup_df["atdtmco_ns_pred_old"] = -pd.to_numeric(
    old_history_dedup_df["forecast_model"], errors="coerce"
)

name_to_code = (
    daily_df
    .dropna(subset=["atdtmco_cashdesk_code"])
    .drop_duplicates("atdtmco_cashdesk_name", keep="last")
    .set_index("atdtmco_cashdesk_name")["atdtmco_cashdesk_code"]
    .to_dict()
)
trn_to_code = (
    daily_df
    .dropna(subset=["atdtmco_cashdesk_code", "atdtmco_cashdesk_name_trn"])
    .drop_duplicates("atdtmco_cashdesk_name_trn", keep="last")
    .set_index("atdtmco_cashdesk_name_trn")["atdtmco_cashdesk_code"]
    .to_dict()
)

WEEKDAY_ALIASES = {
    "пн": 0,
    "вт": 1,
    "ср": 2,
    "чт": 3,
    "пт": 4,
    "сб": 5,
    "вс": 6,
}
WEEKDAY_RANGE_RE = re.compile(
    r"(пн|вт|ср|чт|пт|сб|вс)\s*-\s*(пн|вт|ср|чт|пт|сб|вс)",
    flags=re.IGNORECASE,
)
WEEKDAY_TOKEN_RE = re.compile(
    r"(пн|вт|ср|чт|пт|сб|вс)",
    flags=re.IGNORECASE,
)


def parse_open_weekdays(schedule_text: object) -> FrozenSet[int]:
    if schedule_text is None or (
        isinstance(schedule_text, float) and np.isnan(schedule_text)
    ):
        return frozenset()
    text = str(schedule_text).strip().lower()
    if not text or text in {"nan", "none"}:
        return frozenset()
    open_days: Set[int] = set()
    for segment in text.split(";"):
        covered = set()
        for match in WEEKDAY_RANGE_RE.finditer(segment):
            start = WEEKDAY_ALIASES[match.group(1).lower()]
            end = WEEKDAY_ALIASES[match.group(2).lower()]
            if start <= end:
                values = range(start, end + 1)
            else:
                values = list(range(start, 7)) + list(range(0, end + 1))
            open_days.update(values)
            covered.update({match.group(1).lower(), match.group(2).lower()})
        for token in WEEKDAY_TOKEN_RE.findall(segment):
            token = token.lower()
            if token not in covered:
                open_days.add(WEEKDAY_ALIASES[token])
    return frozenset(open_days)


schedule_open_weekdays: Dict[str, FrozenSet[int]] = {}
schedule_parse_failures = 0
for row in schedule_df.itertuples(index=False):
    code = getattr(row, "c_code_dblink", None)
    if code is None or (isinstance(code, float) and np.isnan(code)):
        continue
    code = str(code).strip()
    if not code or code in {"None", "nan", "NaN"}:
        continue
    open_days = parse_open_weekdays(
        getattr(row, "wtimecorp", None)
    ) | parse_open_weekdays(
        getattr(row, "wtimepriv", None)
    )
    if not open_days:
        if pd.notna(getattr(row, "wtimecorp", None)) or pd.notna(
            getattr(row, "wtimepriv", None)
        ):
            schedule_parse_failures += 1
        continue
    schedule_open_weekdays[code] = open_days

holiday_dates: Set[pd.Timestamp] = set(
    holiday_df.loc[
        holiday_df["hol_msk_flg"].eq(1),
        "cld_day_dt",
    ].tolist()
)


def is_cashdesk_open(
    cashdesk_code: object,
    day: pd.Timestamp,
) -> Optional[bool]:
    if cashdesk_code is None or (
        isinstance(cashdesk_code, float) and np.isnan(cashdesk_code)
    ):
        return None
    code = str(cashdesk_code).strip()
    open_days = schedule_open_weekdays.get(code)
    if open_days is None:
        return None
    day = pd.Timestamp(day).normalize()
    if day in holiday_dates:
        return False
    return int(day.dayofweek) in open_days


def is_cashdesk_closed(
    cashdesk_code: object,
    day: pd.Timestamp,
    has_source_row: bool,
) -> bool:
    open_status = is_cashdesk_open(cashdesk_code, day)
    if open_status is None:
        return not bool(has_source_row)
    return not open_status


active_codes = set(name_to_code.values())
matched_codes = active_codes & set(schedule_open_weekdays)
unmatched_codes = active_codes - set(schedule_open_weekdays)

for directory in (
    CALIBRATION_CHECKPOINT_DIR,
    RETRO_CHECKPOINT_DIR,
    REPORT_DIR,
):
    directory.mkdir(parents=True, exist_ok=True)

print("Загружено {:,} дневных строк и {:,} старых прогнозов".format(
    len(daily_df), len(old_history_dedup_df)
))
print(
    "Расписание: {:,} касс с графиком; без графика: {:,}; "
    "не распарсилось строк: {:,}; праздников: {:,}".format(
        len(matched_codes),
        len(unmatched_codes),
        schedule_parse_failures,
        len(holiday_dates),
    )
)

_parser_checks = [
    (
        "Пн-Пт 09:00-18:00, без перерыва",
        frozenset({0, 1, 2, 3, 4}),
    ),
    (
        "Пн-Пт 09:00-18:00, без перерыва; Сб 10:00-16:00, без перерыва",
        frozenset({0, 1, 2, 3, 4, 5}),
    ),
    ("", frozenset()),
]
for sample_text, expected_days in _parser_checks:
    actual_days = parse_open_weekdays(sample_text)
    if actual_days != expected_days:
        raise RuntimeError(
            "Ошибка парсера расписания: {!r} -> {}, ожидалось {}".format(
                sample_text, actual_days, expected_days
            )
        )
print("Парсер расписания: ok")


In [ ]:
@dataclass(frozen=True)
class SarimaConfig:
    order: Tuple[int, int, int]
    seasonal_order: Tuple[int, int, int, int]
    maxiter: int = 200


SARIMA_CONFIG = SarimaConfig(
    order=SARIMA_ORDER,
    seasonal_order=SARIMA_SEASONAL_ORDER,
)


def make_regular_series(
    cashdesk_df: pd.DataFrame,
    value_column: str,
    date_from: pd.Timestamp,
    date_to: pd.Timestamp,
    cashdesk_code: object = None,
) -> pd.Series:
    full_index = pd.date_range(date_from, date_to, freq="D")
    observed = (
        cashdesk_df
        .drop_duplicates("calday", keep="last")
        .set_index("calday")[value_column]
        .astype(float)
    )
    series = observed.reindex(full_index)
    missing_mask = series.isna()
    if missing_mask.any() and cashdesk_code is not None:
        open_flags = [
            is_cashdesk_open(cashdesk_code, day)
            for day in series.index
        ]
        open_zero_mask = missing_mask & pd.Series(
            [flag is True for flag in open_flags],
            index=series.index,
        )
        series.loc[open_zero_mask] = 0.0
    for exclude_start, exclude_end in EXCLUDE_DATE_RANGES:
        excluded_mask = (
            (series.index >= pd.Timestamp(exclude_start))
            & (series.index < pd.Timestamp(exclude_end))
        )
        series.loc[excluded_mask] = np.nan
    series.index.name = "calday"
    return series


def make_future_index(y: pd.Series, steps: int) -> pd.DatetimeIndex:
    return pd.date_range(
        y.index.max() + pd.Timedelta(days=1),
        periods=steps,
        freq="D",
    )


def guarded_sarima_forecast(
    y: pd.Series,
    steps: int,
    config: SarimaConfig,
) -> np.ndarray:
    model = SARIMAX(
        y,
        order=config.order,
        seasonal_order=config.seasonal_order,
        enforce_stationarity=True,
        enforce_invertibility=True,
    )
    with warnings.catch_warnings():
        warnings.filterwarnings(
            "ignore",
            message="Non-invertible starting.*",
            category=UserWarning,
        )
        warnings.filterwarnings(
            "ignore",
            message="Non-stationary starting.*",
            category=UserWarning,
        )
        warnings.filterwarnings(
            "ignore",
            message="Maximum Likelihood optimization failed.*",
        )
        fitted = model.fit(disp=False, maxiter=config.maxiter)
    if not bool(fitted.mle_retvals.get("converged", False)):
        raise ValueError("SARIMA не сошлась")
    for root_name, roots in (("AR", fitted.arroots), ("MA", fitted.maroots)):
        root_modulus = np.abs(np.asarray(roots, dtype=complex))
        if root_modulus.size and (
            not np.isfinite(root_modulus).all()
            or (root_modulus <= 1.0).any()
        ):
            raise ValueError("Неустойчивые {}-корни".format(root_name))
    forecast = fitted.get_forecast(
        steps=steps
    ).predicted_mean.to_numpy(dtype=float)
    valid_history = y.dropna().to_numpy(dtype=float)
    history_scale = max(
        1.0,
        float(np.quantile(np.abs(valid_history), 0.99)),
    )
    if not np.isfinite(forecast).all():
        raise ValueError("Нечисловой SARIMA-прогноз")
    if (
        np.abs(forecast)
        > FORECAST_SCALE_MULTIPLIER * history_scale
    ).any():
        raise ValueError("Несоразмерный SARIMA-прогноз")
    return forecast


def weekday_point_forecast(
    y: pd.Series,
    steps: int,
    clip_upper_zero: bool,
) -> np.ndarray:
    y_clean = y.dropna()
    if y_clean.empty:
        raise ValueError("Нет истории для fallback")
    global_value = float(y_clean.median())
    weekday_values = y_clean.groupby(y_clean.index.dayofweek).median()
    values = np.asarray([
        weekday_values.get(day.dayofweek, global_value)
        for day in make_future_index(y, steps)
    ], dtype=float)
    if clip_upper_zero:
        values = np.minimum(values, CASH_NEED_CLIP_UPPER)
    return values


def forecast_with_fallback(
    y: pd.Series,
    steps: int,
    clip_upper_zero: bool,
) -> Tuple[np.ndarray, bool]:
    try:
        if y.notna().sum() < MIN_SARIMA_OBS:
            raise ValueError("Недостаточно наблюдений")
        values = guarded_sarima_forecast(y, steps, SARIMA_CONFIG)
        if clip_upper_zero:
            values = np.minimum(values, CASH_NEED_CLIP_UPPER)
        return values, False
    except Exception:
        return weekday_point_forecast(
            y,
            steps,
            clip_upper_zero,
        ), True


def build_scoring_tasks(
    score_date: pd.Timestamp,
) -> Tuple[pd.Timestamp, pd.Timestamp, List[Tuple[int, str, pd.DataFrame]]]:
    score_date = pd.Timestamp(score_date).normalize()
    report_date = score_date - pd.Timedelta(days=DATA_LAG_DAYS)
    history_date_from = (
        report_date
        - pd.DateOffset(months=HISTORY_MONTHS)
        + pd.Timedelta(days=1)
    )
    active_date_from = report_date - pd.DateOffset(
        months=ACTIVE_LOOKBACK_MONTHS
    )
    available_df = daily_df[
        (daily_df["calday"] >= history_date_from)
        & (daily_df["calday"] <= report_date)
    ]
    active_cashdesks = sorted(
        available_df.loc[
            available_df["calday"] >= active_date_from,
            "atdtmco_cashdesk_name",
        ].dropna().unique().tolist()
    )
    tasks = []
    for cashdesk_index, cashdesk_name in enumerate(active_cashdesks):
        cashdesk_df = available_df[
            available_df["atdtmco_cashdesk_name"].eq(cashdesk_name)
        ].copy()
        tasks.append((cashdesk_index, cashdesk_name, cashdesk_df))
    return report_date, history_date_from, tasks


def build_global_scales(
    report_date: pd.Timestamp,
    windows: List[int],
) -> Dict[int, float]:
    scales = {}
    for window in windows:
        date_from = report_date - pd.Timedelta(days=window - 1)
        values = daily_df.loc[
            (daily_df["calday"] >= date_from)
            & (daily_df["calday"] <= report_date)
            & (daily_df["atdtmco_ns_fact"] < 0),
            "atdtmco_ns_fact",
        ].abs()
        scale = float(values.median())
        if not np.isfinite(scale) or scale <= 0:
            raise RuntimeError(
                "Не определён глобальный scale для {} и окна {}".format(
                    report_date.date(), window
                )
            )
        scales[window] = scale
    return scales


def build_cashdesk_scales(
    ns_series: pd.Series,
    report_date: pd.Timestamp,
    windows: List[int],
    global_scales: Dict[int, float],
) -> Dict[int, float]:
    scales = {}
    for window in windows:
        date_from = report_date - pd.Timedelta(days=window - 1)
        values = ns_series[
            (ns_series.index >= date_from) & (ns_series < 0)
        ].abs().dropna()
        scales[window] = (
            float(values.median())
            if len(values) >= MIN_SCALE_ROWS
            else global_scales[window]
        )
    return scales


def weighted_empirical_quantile(
    global_values: np.ndarray,
    cashdesk_values: np.ndarray,
    quantile: float,
    shrinkage: float,
) -> float:
    global_values = np.asarray(global_values, dtype=float)
    cashdesk_values = np.asarray(cashdesk_values, dtype=float)
    global_values = global_values[np.isfinite(global_values)]
    cashdesk_values = cashdesk_values[np.isfinite(cashdesk_values)]
    if len(global_values) == 0:
        raise RuntimeError("Пустой глобальный пул ошибок")
    if len(cashdesk_values) == 0:
        return float(np.quantile(global_values, quantile))
    cashdesk_weight = len(cashdesk_values) / (
        len(cashdesk_values) + shrinkage
    )
    values = np.concatenate([global_values, cashdesk_values])
    weights = np.concatenate([
        np.full(
            len(global_values),
            (1.0 - cashdesk_weight) / len(global_values),
        ),
        np.full(
            len(cashdesk_values),
            cashdesk_weight / len(cashdesk_values),
        ),
    ])
    order = np.argsort(values)
    cumulative_weights = np.cumsum(weights[order])
    index = np.searchsorted(
        cumulative_weights,
        quantile * cumulative_weights[-1],
        side="left",
    )
    return float(values[order][min(index, len(values) - 1)])


def mae95_ns(
    evaluation_df: pd.DataFrame,
    fact_col: str,
    pred_col: str,
) -> float:
    clean_df = evaluation_df[[fact_col, pred_col]].dropna().copy()
    trim_count = int(np.floor(len(clean_df) * 0.05))
    if trim_count > 0:
        trim_index = clean_df.nsmallest(trim_count, fact_col).index
        clean_df = clean_df.drop(index=trim_index)
    return float((clean_df[fact_col] - clean_df[pred_col]).abs().mean())


def mae95_absolute_fact(
    evaluation_df: pd.DataFrame,
    fact_col: str,
    pred_col: str,
) -> float:
    clean_df = evaluation_df[[fact_col, pred_col]].dropna().copy()
    trim_count = int(np.floor(len(clean_df) * 0.05))
    if trim_count > 0:
        trim_index = clean_df[fact_col].abs().nlargest(trim_count).index
        clean_df = clean_df.drop(index=trim_index)
    return float((clean_df[fact_col] - clean_df[pred_col]).abs().mean())


In [ ]:
CALIBRATION_COLUMNS = [
    "score_date",
    "report_date",
    "atdtmco_cashdesk_name",
    "atdtmco_cashdesk_name_trn",
    "forecast_date",
    "atdtmco_ns_pred_central",
    "fallback NS",
] + ["scale_{}".format(window) for window in ERROR_WINDOW_GRID]


def forecast_calibration_cashdesk(
    score_date: pd.Timestamp,
    report_date: pd.Timestamp,
    history_date_from: pd.Timestamp,
    cashdesk_name: str,
    cashdesk_df: pd.DataFrame,
    global_scales: Dict[int, float],
) -> pd.DataFrame:
    cashdesk_code = name_to_code.get(cashdesk_name)
    if cashdesk_code is None and "atdtmco_cashdesk_code" in cashdesk_df:
        codes = cashdesk_df["atdtmco_cashdesk_code"].dropna()
        cashdesk_code = codes.iloc[-1] if len(codes) else None
    ns_series = make_regular_series(
        cashdesk_df,
        "atdtmco_ns_fact",
        history_date_from,
        report_date,
        cashdesk_code=cashdesk_code,
    )
    try:
        central_values, fallback_ns = forecast_with_fallback(
            ns_series,
            MODEL_STEPS,
            True,
        )
    except Exception as error:
        raise RuntimeError(
            "Не построен calibration-прогноз для {} на {}".format(
                cashdesk_name,
                score_date.date(),
            )
        ) from error
    scales = build_cashdesk_scales(
        ns_series,
        report_date,
        ERROR_WINDOW_GRID,
        global_scales,
    )
    translated_names = cashdesk_df[
        "atdtmco_cashdesk_name_trn"
    ].dropna()
    translated_name = (
        translated_names.iloc[-1]
        if len(translated_names) > 0
        else pd.NA
    )
    result_df = pd.DataFrame({
        "forecast_date": make_future_index(ns_series, MODEL_STEPS),
        "atdtmco_ns_pred_central": central_values,
    })
    result_df = result_df[
        (result_df["forecast_date"] >= score_date)
        & (
            result_df["forecast_date"]
            < score_date + pd.Timedelta(days=FORECAST_DAYS)
        )
    ].copy()
    result_df.insert(0, "score_date", score_date)
    result_df.insert(1, "report_date", report_date)
    result_df.insert(2, "atdtmco_cashdesk_name", cashdesk_name)
    result_df.insert(3, "atdtmco_cashdesk_name_trn", translated_name)
    result_df["fallback NS"] = bool(fallback_ns)
    for window, scale in scales.items():
        result_df["scale_{}".format(window)] = scale
    return result_df[CALIBRATION_COLUMNS]


def validate_calibration_result(
    score_date: pd.Timestamp,
    expected_cashdesk_names: List[str],
    result_df: pd.DataFrame,
) -> None:
    if result_df.empty:
        raise ValueError("Пустой calibration checkpoint")
    if list(result_df.columns) != CALIBRATION_COLUMNS:
        raise ValueError("Неверная схема calibration checkpoint")
    if not result_df["score_date"].eq(score_date).all():
        raise ValueError("Calibration checkpoint содержит другую дату")
    actual_cashdesk_names = set(
        result_df["atdtmco_cashdesk_name"].dropna().unique()
    )
    if actual_cashdesk_names != set(expected_cashdesk_names):
        raise ValueError(
            "Calibration checkpoint содержит другой набор касс"
        )
    key_columns = [
        "score_date",
        "atdtmco_cashdesk_name",
        "forecast_date",
    ]
    if result_df.duplicated(key_columns).any():
        raise ValueError("Дубли в calibration checkpoint")
    horizon_counts = result_df.groupby(
        "atdtmco_cashdesk_name"
    )["forecast_date"].nunique()
    if not horizon_counts.eq(FORECAST_DAYS).all():
        raise ValueError("Неверный calibration-горизонт")
    numeric_columns = [
        "atdtmco_ns_pred_central",
    ] + ["scale_{}".format(window) for window in ERROR_WINDOW_GRID]
    if not np.isfinite(
        result_df[numeric_columns].to_numpy(dtype=float)
    ).all():
        raise ValueError("Нечисловые calibration-прогнозы")


def calibration_checkpoint_path(score_date: pd.Timestamp) -> Path:
    return CALIBRATION_CHECKPOINT_DIR / "{}.parquet".format(
        pd.Timestamp(score_date).strftime("%Y-%m-%d")
    )


calibration_parts = []
with parallel_backend("loky", inner_max_num_threads=1):
    with Parallel(n_jobs=N_JOBS) as parallel:
        for score_index, score_date in enumerate(
            calibration_score_dates,
            start=1,
        ):
            score_date = pd.Timestamp(score_date).normalize()
            started_at = time.perf_counter()
            report_date, history_date_from, tasks = build_scoring_tasks(
                score_date
            )
            if not tasks:
                print(
                    "[{}/{}] {}: пропуск — нет активных касс".format(
                        score_index,
                        len(calibration_score_dates),
                        score_date.date(),
                    )
                )
                continue
            expected_cashdesk_names = [task[1] for task in tasks]
            checkpoint_path = calibration_checkpoint_path(score_date)
            if USE_CHECKPOINTS and checkpoint_path.exists():
                day_result_df = pd.read_parquet(checkpoint_path)
                validate_calibration_result(
                    score_date,
                    expected_cashdesk_names,
                    day_result_df,
                )
                source_label = "checkpoint"
            else:
                global_scales = build_global_scales(
                    report_date,
                    ERROR_WINDOW_GRID,
                )
                task_results = parallel(
                    delayed(forecast_calibration_cashdesk)(
                        score_date,
                        report_date,
                        history_date_from,
                        cashdesk_name,
                        cashdesk_df,
                        global_scales,
                    )
                    for _, cashdesk_name, cashdesk_df in tasks
                )
                day_result_df = pd.concat(
                    task_results,
                    ignore_index=True,
                )
                validate_calibration_result(
                    score_date,
                    expected_cashdesk_names,
                    day_result_df,
                )
                temporary_path = checkpoint_path.with_suffix(
                    ".tmp.parquet"
                )
                day_result_df.to_parquet(temporary_path, index=False)
                os.replace(str(temporary_path), str(checkpoint_path))
                source_label = "расчёт"
            calibration_parts.append(day_result_df)
            print(
                "[{}/{}] {}: {} касс, {:.1f} сек, {}".format(
                    score_index,
                    len(calibration_score_dates),
                    score_date.date(),
                    day_result_df[
                        "atdtmco_cashdesk_name"
                    ].nunique(),
                    time.perf_counter() - started_at,
                    source_label,
                )
            )

calibration_forecasts_df = pd.concat(
    calibration_parts,
    ignore_index=True,
)
calibration_forecasts_df["шаг прогноза"] = (
    calibration_forecasts_df["forecast_date"]
    - calibration_forecasts_df["score_date"]
).dt.days.astype(int)
calibration_forecasts_df = calibration_forecasts_df.merge(
    daily_df[[
        "atdtmco_cashdesk_name",
        "calday",
        "atdtmco_ns_fact",
    ]].rename(columns={"calday": "forecast_date"}),
    on=["atdtmco_cashdesk_name", "forecast_date"],
    how="left",
    validate="many_to_one",
)
calibration_forecasts_df["ошибка факт − прогноз"] = (
    calibration_forecasts_df["atdtmco_ns_fact"]
    - calibration_forecasts_df["atdtmco_ns_pred_central"]
)
calibration_error_candidates_df = calibration_forecasts_df[
    (calibration_forecasts_df["forecast_date"] <= PRETEST_REPORT_DATE)
    & (calibration_forecasts_df["atdtmco_ns_fact"] < 0)
].copy()
selection_first_day_df = calibration_forecasts_df[
    calibration_forecasts_df["score_date"].isin(
        pretest_selection_dates
    )
    & calibration_forecasts_df["forecast_date"].eq(
        calibration_forecasts_df["score_date"]
    )
    & (calibration_forecasts_df["atdtmco_ns_fact"] < 0)
].copy()

calibration_forecasts_df.to_parquet(
    OUTPUT_DIR / "calibration_forecasts.parquet",
    index=False,
)
print("Calibration: {:,} прогнозных строк, {:,} ошибок".format(
    len(calibration_forecasts_df),
    len(calibration_error_candidates_df),
))


In [ ]:
def simulate_pretest_candidate(
    window: int,
    shrinkage: float,
) -> Tuple[Dict[str, object], pd.DataFrame]:
    scale_column = "scale_{}".format(window)
    error_source_df = calibration_error_candidates_df.copy()
    error_source_df["нормированная ошибка"] = (
        error_source_df["ошибка факт − прогноз"]
        / error_source_df[scale_column]
    )
    adaptive_quantile = float(INITIAL_QUANTILE)
    prediction_parts = []

    for score_date in pretest_selection_dates:
        score_date = pd.Timestamp(score_date).normalize()
        report_date = score_date - pd.Timedelta(days=DATA_LAG_DAYS)
        if prediction_parts:
            previous_df = pd.concat(prediction_parts, ignore_index=True)
            newly_known_df = previous_df[
                previous_df["score_date"].eq(report_date)
                & (previous_df["atdtmco_ns_fact"] < 0)
            ]
            if len(newly_known_df) >= ADAPTIVE_MIN_UPDATE_ROWS:
                observed_breach = float((
                    newly_known_df["atdtmco_ns_fact"]
                    < newly_known_df["прогноз кандидата"]
                ).mean())
                adaptive_quantile = float(np.clip(
                    adaptive_quantile
                    + ADAPTIVE_GAMMA
                    * (TARGET_BREACH_RATE - observed_breach),
                    ADAPTIVE_QUANTILE_MIN,
                    ADAPTIVE_QUANTILE_MAX,
                ))

        current_df = selection_first_day_df[
            selection_first_day_df["score_date"].eq(score_date)
        ].copy()
        if current_df.empty:
            continue
        error_date_from = report_date - pd.Timedelta(days=window - 1)
        available_errors_df = error_source_df[
            (error_source_df["forecast_date"] >= error_date_from)
            & (error_source_df["forecast_date"] <= report_date)
            & error_source_df["шаг прогноза"].eq(0)
        ]
        global_errors = available_errors_df[
            "нормированная ошибка"
        ].to_numpy()
        if len(global_errors) == 0:
            raise RuntimeError(
                "Нет предтестовых ошибок для {} и окна {}".format(
                    score_date.date(), window
                )
            )
        cashdesk_errors = {
            cashdesk_name: cashdesk_df[
                "нормированная ошибка"
            ].to_numpy()
            for cashdesk_name, cashdesk_df in available_errors_df.groupby(
                "atdtmco_cashdesk_name"
            )
        }
        current_df["поправка кандидата"] = current_df[
            "atdtmco_cashdesk_name"
        ].map({
            cashdesk_name: weighted_empirical_quantile(
                global_errors,
                cashdesk_errors.get(
                    cashdesk_name,
                    np.array([], dtype=float),
                ),
                adaptive_quantile,
                shrinkage,
            )
            for cashdesk_name in current_df[
                "atdtmco_cashdesk_name"
            ].unique()
        })
        current_df["использованный квантиль"] = adaptive_quantile
        current_df["прогноз кандидата"] = np.minimum(
            current_df["atdtmco_ns_pred_central"]
            + current_df["поправка кандидата"]
            * current_df[scale_column],
            CASH_NEED_CLIP_UPPER,
        )
        prediction_parts.append(current_df)

    predictions_df = pd.concat(prediction_parts, ignore_index=True)
    fact = predictions_df["atdtmco_ns_fact"]
    prediction = predictions_df["прогноз кандидата"]
    breach_rate = float((fact < prediction).mean())
    metrics = {
        "окно, дней": window,
        "сила стягивания": shrinkage,
        "количество строк": len(predictions_df),
        "% невыдач": breach_rate,
        "отклонение от диапазона 13–15%": max(
            0.13 - breach_rate,
            breach_rate - 0.15,
            0.0,
        ),
        "MAE": float((fact - prediction).abs().mean()),
        "MAE95": mae95_ns(
            predictions_df,
            "atdtmco_ns_fact",
            "прогноз кандидата",
        ),
        "уровень 13–15%": 0.13 <= breach_rate <= 0.15,
    }
    return metrics, predictions_df


window_candidate_rows = []
pretest_candidate_predictions = {}
for window in ERROR_WINDOW_GRID:
    for shrinkage in SHRINKAGE_GRID:
        candidate_metrics, candidate_predictions_df = (
            simulate_pretest_candidate(window, shrinkage)
        )
        window_candidate_rows.append(candidate_metrics)
        pretest_candidate_predictions[(window, shrinkage)] = (
            candidate_predictions_df
        )

window_candidate_comparison_df = pd.DataFrame(window_candidate_rows)
window_candidate_ranking_df = window_candidate_comparison_df.sort_values(
    [
        "отклонение от диапазона 13–15%",
        "MAE95",
        "MAE",
    ],
    ascending=True,
).reset_index(drop=True)
selected_error_window = int(
    window_candidate_ranking_df.iloc[0]["окно, дней"]
)
selected_shrinkage = float(
    window_candidate_ranking_df.iloc[0]["сила стягивания"]
)
window_comparison_df = (
    window_candidate_comparison_df
    .sort_values(
        [
            "окно, дней",
            "отклонение от диапазона 13–15%",
            "MAE95",
            "MAE",
        ]
    )
    .groupby("окно, дней", as_index=False)
    .first()
    .sort_values("окно, дней")
    .reset_index(drop=True)
)
selected_pretest_predictions_df = pretest_candidate_predictions[
    (selected_error_window, selected_shrinkage)
]

window_candidate_comparison_df.to_csv(
    REPORT_DIR / "all_window_candidates.csv",
    index=False,
)
window_comparison_df.to_csv(
    REPORT_DIR / "window_comparison.csv",
    index=False,
)
window_comparison_df.to_parquet(
    REPORT_DIR / "window_comparison.parquet",
    index=False,
)

print(
    "Выбраны окно {} дней и сила стягивания {:.0f}".format(
        selected_error_window,
        selected_shrinkage,
    )
)
display(window_comparison_df.style.format({
    "сила стягивания": "{:.0f}",
    "% невыдач": "{:.2%}",
    "отклонение от диапазона 13–15%": "{:.2%}",
    "MAE": "{:,.0f}",
    "MAE95": "{:,.0f}",
}))

In [ ]:
SELECTED_RETRO_CHECKPOINT_DIR = (
    RETRO_CHECKPOINT_DIR
    / "window_{}".format(selected_error_window)
)
SELECTED_RETRO_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

RETRO_COLUMNS = [
    "score_date",
    "report_date",
    "atdtmco_cashdesk_name",
    "atdtmco_cashdesk_name_trn",
    "forecast_date",
    "atdtmco_ns_pred_central",
    "atdtmco_saldo_turn_pred",
    "нормирующий масштаб",
    "fallback NS",
    "fallback saldo",
]


def forecast_retro_cashdesk(
    score_date: pd.Timestamp,
    report_date: pd.Timestamp,
    history_date_from: pd.Timestamp,
    cashdesk_name: str,
    cashdesk_df: pd.DataFrame,
    global_scale: float,
) -> pd.DataFrame:
    cashdesk_code = name_to_code.get(cashdesk_name)
    if cashdesk_code is None and "atdtmco_cashdesk_code" in cashdesk_df:
        codes = cashdesk_df["atdtmco_cashdesk_code"].dropna()
        cashdesk_code = codes.iloc[-1] if len(codes) else None
    ns_series = make_regular_series(
        cashdesk_df,
        "atdtmco_ns_fact",
        history_date_from,
        report_date,
        cashdesk_code=cashdesk_code,
    )
    saldo_series = make_regular_series(
        cashdesk_df,
        "atdtmco_saldo_turn_fact",
        history_date_from,
        report_date,
        cashdesk_code=cashdesk_code,
    )
    try:
        ns_values, fallback_ns = forecast_with_fallback(
            ns_series,
            MODEL_STEPS,
            True,
        )
    except Exception as error:
        raise RuntimeError(
            "Не построен retro NS-прогноз для {} на {}".format(
                cashdesk_name,
                score_date.date(),
            )
        ) from error
    try:
        saldo_values, fallback_saldo = forecast_with_fallback(
            saldo_series,
            MODEL_STEPS,
            False,
        )
    except Exception:
        saldo_values = np.zeros(MODEL_STEPS, dtype=float)
        fallback_saldo = True

    scale = build_cashdesk_scales(
        ns_series,
        report_date,
        [selected_error_window],
        {selected_error_window: global_scale},
    )[selected_error_window]
    translated_names = cashdesk_df[
        "atdtmco_cashdesk_name_trn"
    ].dropna()
    translated_name = (
        translated_names.iloc[-1]
        if len(translated_names) > 0
        else pd.NA
    )
    result_df = pd.DataFrame({
        "forecast_date": make_future_index(ns_series, MODEL_STEPS),
        "atdtmco_ns_pred_central": ns_values,
        "atdtmco_saldo_turn_pred": saldo_values,
    })
    result_df = result_df[
        (result_df["forecast_date"] >= score_date)
        & (
            result_df["forecast_date"]
            < score_date + pd.Timedelta(days=FORECAST_DAYS)
        )
    ].copy()
    result_df.insert(0, "score_date", score_date)
    result_df.insert(1, "report_date", report_date)
    result_df.insert(2, "atdtmco_cashdesk_name", cashdesk_name)
    result_df.insert(3, "atdtmco_cashdesk_name_trn", translated_name)
    result_df["нормирующий масштаб"] = scale
    result_df["fallback NS"] = bool(fallback_ns)
    result_df["fallback saldo"] = bool(fallback_saldo)
    return result_df[RETRO_COLUMNS]


def validate_retro_result(
    score_date: pd.Timestamp,
    expected_cashdesk_names: List[str],
    result_df: pd.DataFrame,
) -> None:
    if result_df.empty:
        raise ValueError("Пустой retro checkpoint")
    if list(result_df.columns) != RETRO_COLUMNS:
        raise ValueError("Неверная схема retro checkpoint")
    if not result_df["score_date"].eq(score_date).all():
        raise ValueError("Retro checkpoint содержит другую дату")
    actual_cashdesk_names = set(
        result_df["atdtmco_cashdesk_name"].dropna().unique()
    )
    if actual_cashdesk_names != set(expected_cashdesk_names):
        raise ValueError("Retro checkpoint содержит другой набор касс")
    key_columns = [
        "score_date",
        "atdtmco_cashdesk_name",
        "forecast_date",
    ]
    if result_df.duplicated(key_columns).any():
        raise ValueError("Дубли в retro checkpoint")
    horizon_counts = result_df.groupby(
        "atdtmco_cashdesk_name"
    )["forecast_date"].nunique()
    if not horizon_counts.eq(FORECAST_DAYS).all():
        raise ValueError("Неверный retro-горизонт")
    numeric_columns = [
        "atdtmco_ns_pred_central",
        "atdtmco_saldo_turn_pred",
        "нормирующий масштаб",
    ]
    if not np.isfinite(
        result_df[numeric_columns].to_numpy(dtype=float)
    ).all():
        raise ValueError("Нечисловые retro-прогнозы")


def retro_checkpoint_path(score_date: pd.Timestamp) -> Path:
    return SELECTED_RETRO_CHECKPOINT_DIR / "{}.parquet".format(
        pd.Timestamp(score_date).strftime("%Y-%m-%d")
    )


retro_parts = []
with parallel_backend("loky", inner_max_num_threads=1):
    with Parallel(n_jobs=N_JOBS) as parallel:
        for score_index, score_date in enumerate(score_dates, start=1):
            score_date = pd.Timestamp(score_date).normalize()
            started_at = time.perf_counter()
            report_date, history_date_from, tasks = build_scoring_tasks(
                score_date
            )
            expected_cashdesk_names = [task[1] for task in tasks]
            checkpoint_path = retro_checkpoint_path(score_date)
            if USE_CHECKPOINTS and checkpoint_path.exists():
                day_result_df = pd.read_parquet(checkpoint_path)
                validate_retro_result(
                    score_date,
                    expected_cashdesk_names,
                    day_result_df,
                )
                source_label = "checkpoint"
            else:
                global_scale = build_global_scales(
                    report_date,
                    [selected_error_window],
                )[selected_error_window]
                task_results = parallel(
                    delayed(forecast_retro_cashdesk)(
                        score_date,
                        report_date,
                        history_date_from,
                        cashdesk_name,
                        cashdesk_df,
                        global_scale,
                    )
                    for _, cashdesk_name, cashdesk_df in tasks
                )
                day_result_df = pd.concat(
                    task_results,
                    ignore_index=True,
                )
                validate_retro_result(
                    score_date,
                    expected_cashdesk_names,
                    day_result_df,
                )
                temporary_path = checkpoint_path.with_suffix(
                    ".tmp.parquet"
                )
                day_result_df.to_parquet(temporary_path, index=False)
                os.replace(str(temporary_path), str(checkpoint_path))
                source_label = "расчёт"
            retro_parts.append(day_result_df)
            print(
                "[{}/{}] {}: {} касс, {:.1f} сек, {}".format(
                    score_index,
                    len(score_dates),
                    score_date.date(),
                    day_result_df[
                        "atdtmco_cashdesk_name"
                    ].nunique(),
                    time.perf_counter() - started_at,
                    source_label,
                )
            )

retro_central_df = pd.concat(retro_parts, ignore_index=True)
actual_daily_df = daily_df[[
    "atdtmco_cashdesk_name",
    "calday",
    "atdtmco_cashdesk_name_trn",
    "atdtmco_cashdesk_code",
    "atdtmco_ns_fact",
    "atdtmco_saldo_turn_fact",
]].rename(columns={
    "calday": "forecast_date",
    "atdtmco_cashdesk_name_trn": "atdtmco_cashdesk_name_trn_actual",
    "atdtmco_cashdesk_code": "atdtmco_cashdesk_code_actual",
})
actual_daily_df["есть исходная строка"] = True
retro_central_df = retro_central_df.merge(
    actual_daily_df,
    on=["atdtmco_cashdesk_name", "forecast_date"],
    how="left",
    validate="many_to_one",
)
retro_central_df["atdtmco_cashdesk_name_trn"] = retro_central_df[
    "atdtmco_cashdesk_name_trn_actual"
].combine_first(retro_central_df["atdtmco_cashdesk_name_trn"])
retro_central_df["atdtmco_cashdesk_code"] = retro_central_df[
    "atdtmco_cashdesk_code_actual"
].combine_first(
    retro_central_df["atdtmco_cashdesk_name"].map(name_to_code)
)
retro_central_df = retro_central_df.drop(
    columns=[
        "atdtmco_cashdesk_name_trn_actual",
        "atdtmco_cashdesk_code_actual",
    ]
)
has_source_row = retro_central_df["есть исходная строка"].fillna(False).astype(bool)
retro_central_df["касса закрыта"] = [
    is_cashdesk_closed(code, day, has_row)
    for code, day, has_row in zip(
        retro_central_df["atdtmco_cashdesk_code"],
        retro_central_df["forecast_date"],
        has_source_row,
    )
]
open_without_row = (~has_source_row) & (~retro_central_df["касса закрыта"])
retro_central_df = retro_central_df.drop(columns="есть исходная строка")
retro_central_df.loc[
    open_without_row,
    ["atdtmco_ns_fact", "atdtmco_saldo_turn_fact"],
] = 0.0
retro_central_df.loc[
    retro_central_df["касса закрыта"],
    ["atdtmco_ns_fact", "atdtmco_saldo_turn_fact"],
] = 0.0
retro_central_df.loc[
    retro_central_df["касса закрыта"],
    ["atdtmco_ns_pred_central", "atdtmco_saldo_turn_pred"],
] = 0.0
print(
    "Закрытые дни: {:,}; открытые без операций: {:,}".format(
        int(retro_central_df["касса закрыта"].sum()),
        int(open_without_row.sum()),
    )
)
retro_central_df["шаг прогноза"] = (
    retro_central_df["forecast_date"]
    - retro_central_df["score_date"]
).dt.days.astype(int)
retro_central_df.to_parquet(
    OUTPUT_DIR / "retro_central.parquet",
    index=False,
)
print("Ретро central: {:,} строк".format(len(retro_central_df)))


In [ ]:
selected_scale_column = "scale_{}".format(selected_error_window)
calibration_dynamic_error_df = calibration_error_candidates_df.copy()
calibration_dynamic_error_df["нормированная ошибка"] = (
    calibration_dynamic_error_df["ошибка факт − прогноз"]
    / calibration_dynamic_error_df[selected_scale_column]
)
calibration_dynamic_error_df = calibration_dynamic_error_df[[
    "atdtmco_cashdesk_name",
    "forecast_date",
    "шаг прогноза",
    "нормированная ошибка",
]]

retro_dynamic_error_df = retro_central_df[
    retro_central_df["atdtmco_ns_fact"] < 0
].copy()
retro_dynamic_error_df["нормированная ошибка"] = (
    retro_dynamic_error_df["atdtmco_ns_fact"]
    - retro_dynamic_error_df["atdtmco_ns_pred_central"]
) / retro_dynamic_error_df["нормирующий масштаб"]
retro_dynamic_error_df = retro_dynamic_error_df[[
    "atdtmco_cashdesk_name",
    "forecast_date",
    "шаг прогноза",
    "нормированная ошибка",
]]

ADAPTIVE_COLUMNS = [
    "score_date",
    "atdtmco_cashdesk_name",
    "forecast_date",
    "использованный квантиль",
    "иерархическая поправка",
    "atdtmco_ns_pred",
    "факт для обновления",
]


def calculate_adaptive_date(
    score_date: pd.Timestamp,
    adaptive_quantile: float,
) -> pd.DataFrame:
    score_date = pd.Timestamp(score_date).normalize()
    report_date = score_date - pd.Timedelta(days=DATA_LAG_DAYS)
    error_date_from = report_date - pd.Timedelta(
        days=selected_error_window - 1
    )
    available_errors_df = pd.concat([
        calibration_dynamic_error_df[
            (calibration_dynamic_error_df["forecast_date"] >= error_date_from)
            & (calibration_dynamic_error_df["forecast_date"] <= report_date)
        ],
        retro_dynamic_error_df[
            (retro_dynamic_error_df["forecast_date"] >= error_date_from)
            & (retro_dynamic_error_df["forecast_date"] <= report_date)
        ],
    ], ignore_index=True)
    if available_errors_df.empty:
        raise RuntimeError(
            "Нет доступных ошибок для {}".format(score_date.date())
        )

    current_df = retro_central_df[
        retro_central_df["score_date"].eq(score_date)
    ].copy()
    all_available_errors = available_errors_df[
        "нормированная ошибка"
    ].to_numpy()
    correction_rows = []
    for forecast_step, step_current_df in current_df.groupby(
        "шаг прогноза",
        sort=True,
    ):
        step_errors_df = available_errors_df[
            available_errors_df["шаг прогноза"].eq(forecast_step)
        ]
        step_global_errors = step_errors_df[
            "нормированная ошибка"
        ].to_numpy()
        if len(step_global_errors) == 0:
            step_global_errors = all_available_errors
        step_cashdesk_errors = {
            cashdesk_name: cashdesk_df[
                "нормированная ошибка"
            ].to_numpy()
            for cashdesk_name, cashdesk_df in step_errors_df.groupby(
                "atdtmco_cashdesk_name"
            )
        }
        for cashdesk_name in step_current_df[
            "atdtmco_cashdesk_name"
        ].unique():
            correction_rows.append({
                "atdtmco_cashdesk_name": cashdesk_name,
                "шаг прогноза": forecast_step,
                "иерархическая поправка": weighted_empirical_quantile(
                    step_global_errors,
                    step_cashdesk_errors.get(
                        cashdesk_name,
                        np.array([], dtype=float),
                    ),
                    adaptive_quantile,
                    selected_shrinkage,
                ),
            })
    current_df = current_df.merge(
        pd.DataFrame(correction_rows),
        on=["atdtmco_cashdesk_name", "шаг прогноза"],
        how="left",
        validate="many_to_one",
    )
    current_df["использованный квантиль"] = adaptive_quantile
    current_df["atdtmco_ns_pred"] = np.minimum(
        current_df["atdtmco_ns_pred_central"]
        + current_df["иерархическая поправка"]
        * current_df["нормирующий масштаб"],
        CASH_NEED_CLIP_UPPER,
    )
    current_df.loc[
        current_df["касса закрыта"],
        "atdtmco_ns_pred",
    ] = 0.0
    current_df["факт для обновления"] = current_df[
        "atdtmco_ns_fact"
    ]
    result_df = current_df[ADAPTIVE_COLUMNS].copy()
    if not np.isfinite(result_df[[
        "использованный квантиль",
        "иерархическая поправка",
        "atdtmco_ns_pred",
    ]].to_numpy(dtype=float)).all():
        raise RuntimeError(
            "Нечисловой адаптивный прогноз для {}".format(
                score_date.date()
            )
        )
    return result_df


adaptive_quantile = float(INITIAL_QUANTILE)
adaptive_parts = []
adaptive_first_day_parts = []
adaptive_update_rows = []

for score_index, score_date in enumerate(score_dates, start=1):
    score_date = pd.Timestamp(score_date).normalize()
    report_date = score_date - pd.Timedelta(days=DATA_LAG_DAYS)
    if adaptive_first_day_parts:
        first_day_history_df = pd.concat(
            adaptive_first_day_parts,
            ignore_index=True,
        )
        newly_known_df = first_day_history_df[
            first_day_history_df["score_date"].eq(report_date)
            & (first_day_history_df["факт для обновления"] < 0)
        ]
        if len(newly_known_df) >= ADAPTIVE_MIN_UPDATE_ROWS:
            observed_breach_rate = float((
                newly_known_df["факт для обновления"]
                < newly_known_df["atdtmco_ns_pred"]
            ).mean())
            quantile_before = adaptive_quantile
            adaptive_quantile = float(np.clip(
                adaptive_quantile
                + ADAPTIVE_GAMMA
                * (TARGET_BREACH_RATE - observed_breach_rate),
                ADAPTIVE_QUANTILE_MIN,
                ADAPTIVE_QUANTILE_MAX,
            ))
            adaptive_update_rows.append({
                "дата доступного факта": report_date,
                "количество строк": len(newly_known_df),
                "% невыдач": observed_breach_rate,
                "квантиль до обновления": quantile_before,
                "квантиль после обновления": adaptive_quantile,
            })

    day_result_df = calculate_adaptive_date(
        score_date,
        adaptive_quantile,
    )
    adaptive_parts.append(day_result_df)
    adaptive_first_day_parts.append(day_result_df[
        day_result_df["forecast_date"].eq(score_date)
    ])
    print(
        "[{}/{}] {}: q={:.3f}".format(
            score_index,
            len(score_dates),
            score_date.date(),
            adaptive_quantile,
        )
    )

adaptive_prediction_with_fact_df = pd.concat(
    adaptive_parts,
    ignore_index=True,
)
adaptive_update_history_df = pd.DataFrame(adaptive_update_rows)
adaptive_prediction_df = adaptive_prediction_with_fact_df.drop(
    columns="факт для обновления"
)
final_forecasts_df = retro_central_df.merge(
    adaptive_prediction_df,
    on=["score_date", "atdtmco_cashdesk_name", "forecast_date"],
    how="left",
    validate="one_to_one",
)
if final_forecasts_df["atdtmco_ns_pred"].isna().any():
    raise RuntimeError("Не для всех строк построен итоговый NS-прогноз")

final_forecasts_df.to_parquet(
    OUTPUT_DIR / "final_forecasts.parquet",
    index=False,
)
adaptive_update_history_df.to_csv(
    REPORT_DIR / "adaptive_quantile_history.csv",
    index=False,
)
print("Итоговый прогноз: {:,} строк".format(len(final_forecasts_df)))

In [ ]:
def fill_old_forecast_lookback(
    old_df: pd.DataFrame,
    date_from: pd.Timestamp,
    date_to: pd.Timestamp,
) -> pd.DataFrame:
    date_from = pd.Timestamp(date_from).normalize()
    date_to = pd.Timestamp(date_to).normalize()
    full_index = pd.date_range(date_from, date_to, freq="D")
    filled_parts = []
    lookback_fills = 0
    for cashdesk_name, cashdesk_old_df in old_df.groupby(
        "cashdesk_name",
        sort=False,
    ):
        series = (
            cashdesk_old_df
            .drop_duplicates("calday", keep="last")
            .set_index("calday")["atdtmco_ns_pred_old"]
            .astype(float)
            .reindex(full_index)
        )
        cashdesk_code = trn_to_code.get(cashdesk_name)
        last_nonzero = np.nan
        filled_values = []
        for day, value in series.items():
            numeric_value = (
                float(value) if pd.notna(value) else np.nan
            )
            if pd.notna(numeric_value) and numeric_value != 0.0:
                last_nonzero = numeric_value
            open_status = is_cashdesk_open(cashdesk_code, day)
            if open_status is True and (
                pd.isna(numeric_value) or numeric_value == 0.0
            ):
                if pd.notna(last_nonzero):
                    filled_values.append(last_nonzero)
                    lookback_fills += 1
                else:
                    filled_values.append(numeric_value)
            else:
                filled_values.append(numeric_value)
        filled_parts.append(pd.DataFrame({
            "cashdesk_name": cashdesk_name,
            "calday": full_index,
            "atdtmco_ns_pred_old": filled_values,
        }))
    filled_df = pd.concat(filled_parts, ignore_index=True)
    print(
        "Old lookback: заполнено {:,} открытых дней с 0/пропуском".format(
            lookback_fills
        )
    )
    return filled_df


old_forecast_filled_df = fill_old_forecast_lookback(
    old_history_dedup_df,
    OLD_DATE_FROM,
    RETRO_DATE_TO,
)

first_day_df = final_forecasts_df[
    final_forecasts_df["forecast_date"].eq(
        final_forecasts_df["score_date"]
    )
].copy()
common_first_day_df = first_day_df.merge(
    old_forecast_filled_df[[
        "cashdesk_name",
        "calday",
        "atdtmco_ns_pred_old",
    ]],
    left_on=["atdtmco_cashdesk_name_trn", "forecast_date"],
    right_on=["cashdesk_name", "calday"],
    how="left",
    validate="many_to_one",
)
old_raw_series = (
    old_history_dedup_df
    .drop_duplicates(["cashdesk_name", "calday"], keep="last")
    .set_index(["cashdesk_name", "calday"])["atdtmco_ns_pred_old"]
)
common_first_day_df["atdtmco_ns_pred_old_raw"] = [
    old_raw_series.get((name, day), np.nan)
    for name, day in zip(
        common_first_day_df["atdtmco_cashdesk_name_trn"],
        common_first_day_df["forecast_date"],
    )
]
# Keep open days with fact < 0 OR fact == 0 (zero-need open days).
# Closed days are excluded. Old forecast on zero/missing open days
# comes from lookback to the previous non-zero old prediction.
common_first_day_df = common_first_day_df[
    (~common_first_day_df["касса закрыта"])
    & common_first_day_df["atdtmco_ns_fact"].notna()
    & (common_first_day_df["atdtmco_ns_fact"] <= 0)
    & common_first_day_df["atdtmco_ns_pred_old"].notna()
].reset_index(drop=True)
if common_first_day_df.empty:
    raise RuntimeError("Нет общей NS-выборки по открытым дням")
if common_first_day_df["касса закрыта"].any():
    raise RuntimeError("В NS-выборку попали закрытые дни")
zero_need_rows = int((common_first_day_df["atdtmco_ns_fact"] == 0).sum())
lookback_used_rows = int((
    (common_first_day_df["atdtmco_ns_fact"] == 0)
    & (
        common_first_day_df["atdtmco_ns_pred_old_raw"].isna()
        | (common_first_day_df["atdtmco_ns_pred_old_raw"] == 0)
    )
    & (common_first_day_df["atdtmco_ns_pred_old"] != 0)
).sum())
print(
    "NS-выборка для сравнения: {:,} строк, {:,} касс; "
    "из них потребность=0: {:,}; old lookback на нулях: {:,}".format(
        len(common_first_day_df),
        common_first_day_df["atdtmco_cashdesk_name"].nunique(),
        zero_need_rows,
        lookback_used_rows,
    )
)


def build_ns_metric_row(
    evaluation_df: pd.DataFrame,
    variant_name: str,
    pred_col: str,
) -> Dict[str, object]:
    fact = evaluation_df["atdtmco_ns_fact"]
    prediction = evaluation_df[pred_col]
    error = fact - prediction
    absolute_error = error.abs()
    breach = fact < prediction
    return {
        "вариант": variant_name,
        "количество строк": len(evaluation_df),
        "количество касс": evaluation_df[
            "atdtmco_cashdesk_name"
        ].nunique(),
        "количество невыдач": int(breach.sum()),
        "% невыдач": float(breach.mean()),
        "отклонение от цели": float(
            breach.mean() - TARGET_BREACH_RATE
        ),
        "среднее факта": float(fact.mean()),
        "среднее прогноза": float(prediction.mean()),
        "MAE": float(absolute_error.mean()),
        "MAE95": mae95_ns(
            evaluation_df,
            "atdtmco_ns_fact",
            pred_col,
        ),
        "медианная абсолютная ошибка": float(
            absolute_error.median()
        ),
        "P90 абсолютной ошибки": float(
            absolute_error.quantile(0.90)
        ),
        "смещение факт − прогноз": float(error.mean()),
        "дисперсия ошибок": float(error.var(ddof=1)),
        "СКО ошибок": float(error.std(ddof=1)),
    }


ns_overall_comparison_df = pd.DataFrame([
    build_ns_metric_row(
        common_first_day_df,
        "старый forecast_model",
        "atdtmco_ns_pred_old",
    ),
    build_ns_metric_row(
        common_first_day_df,
        "центральный SARIMA",
        "atdtmco_ns_pred_central",
    ),
    build_ns_metric_row(
        common_first_day_df,
        "итоговый адаптивный алгоритм",
        "atdtmco_ns_pred",
    ),
])


def build_ns_cashdesk_row(
    cashdesk_df: pd.DataFrame,
) -> Dict[str, object]:
    fact = cashdesk_df["atdtmco_ns_fact"]
    old_prediction = cashdesk_df["atdtmco_ns_pred_old"]
    new_prediction = cashdesk_df["atdtmco_ns_pred"]
    old_error = fact - old_prediction
    new_error = fact - new_prediction
    old_absolute_error = old_error.abs()
    new_absolute_error = new_error.abs()
    old_mae = float(old_absolute_error.mean())
    new_mae = float(new_absolute_error.mean())
    old_mae95 = mae95_ns(
        cashdesk_df,
        "atdtmco_ns_fact",
        "atdtmco_ns_pred_old",
    )
    new_mae95 = mae95_ns(
        cashdesk_df,
        "atdtmco_ns_fact",
        "atdtmco_ns_pred",
    )
    old_breach = fact < old_prediction
    new_breach = fact < new_prediction
    translated_names = cashdesk_df[
        "atdtmco_cashdesk_name_trn"
    ].dropna()
    return {
        "касса": cashdesk_df["atdtmco_cashdesk_name"].iloc[0],
        "английское название": (
            translated_names.iloc[-1]
            if len(translated_names) > 0
            else pd.NA
        ),
        "количество строк": len(cashdesk_df),
        "среднее факта": float(fact.mean()),
        "старый: количество невыдач": int(old_breach.sum()),
        "старый: % невыдач": float(old_breach.mean()),
        "новый: количество невыдач": int(new_breach.sum()),
        "новый: % невыдач": float(new_breach.mean()),
        "изменение % невыдач": float(
            new_breach.mean() - old_breach.mean()
        ),
        "старый: среднее прогноза": float(old_prediction.mean()),
        "новый: среднее прогноза": float(new_prediction.mean()),
        "старый: MAE": old_mae,
        "новый: MAE": new_mae,
        "улучшение MAE": old_mae - new_mae,
        "улучшение MAE, %": (
            (old_mae - new_mae) / old_mae
            if old_mae > 0
            else np.nan
        ),
        "старый: MAE95": old_mae95,
        "новый: MAE95": new_mae95,
        "улучшение MAE95": old_mae95 - new_mae95,
        "улучшение MAE95, %": (
            (old_mae95 - new_mae95) / old_mae95
            if old_mae95 > 0
            else np.nan
        ),
        "старый: медианная ошибка": float(
            old_absolute_error.median()
        ),
        "новый: медианная ошибка": float(
            new_absolute_error.median()
        ),
        "старый: P90 ошибки": float(
            old_absolute_error.quantile(0.90)
        ),
        "новый: P90 ошибки": float(
            new_absolute_error.quantile(0.90)
        ),
        "старый: смещение факт − прогноз": float(
            old_error.mean()
        ),
        "новый: смещение факт − прогноз": float(
            new_error.mean()
        ),
        "старый: дисперсия ошибок": float(
            old_error.var(ddof=1)
        ),
        "новый: дисперсия ошибок": float(
            new_error.var(ddof=1)
        ),
        "старый: СКО ошибок": float(old_error.std(ddof=1)),
        "новый: СКО ошибок": float(new_error.std(ddof=1)),
    }


ns_by_cashdesk_df = pd.DataFrame([
    build_ns_cashdesk_row(cashdesk_df)
    for _, cashdesk_df in common_first_day_df.groupby(
        "atdtmco_cashdesk_name",
        sort=True,
    )
]).sort_values(
    ["улучшение MAE, %", "новый: % невыдач"],
    ascending=[True, False],
).reset_index(drop=True)
mae_improved_mask = ns_by_cashdesk_df["улучшение MAE"] > 0
ns_cashdesk_summary_df = pd.DataFrame([{
    "всего касс": len(ns_by_cashdesk_df),
    "касс с улучшением MAE": int(mae_improved_mask.sum()),
    "% касс с улучшением MAE": float(mae_improved_mask.mean()),
    "касс без улучшения MAE": int((~mae_improved_mask).sum()),
}])


def build_dynamic_date_row(date_df: pd.DataFrame) -> Dict[str, object]:
    fact = date_df["atdtmco_ns_fact"]
    old_prediction = date_df["atdtmco_ns_pred_old"]
    new_prediction = date_df["atdtmco_ns_pred"]
    old_error = fact - old_prediction
    new_error = fact - new_prediction
    old_absolute_error = old_error.abs()
    new_absolute_error = new_error.abs()
    old_mae = float(old_absolute_error.mean())
    new_mae = float(new_absolute_error.mean())
    old_mae95 = mae95_ns(
        date_df,
        "atdtmco_ns_fact",
        "atdtmco_ns_pred_old",
    )
    new_mae95 = mae95_ns(
        date_df,
        "atdtmco_ns_fact",
        "atdtmco_ns_pred",
    )
    return {
        "дата скоринга": date_df["score_date"].iloc[0],
        "количество строк": len(date_df),
        "среднее факта": float(fact.mean()),
        "старый: количество невыдач": int(
            (fact < old_prediction).sum()
        ),
        "старый: % невыдач": float((fact < old_prediction).mean()),
        "новый: количество невыдач": int(
            (fact < new_prediction).sum()
        ),
        "новый: % невыдач": float((fact < new_prediction).mean()),
        "изменение % невыдач": float(
            (fact < new_prediction).mean()
            - (fact < old_prediction).mean()
        ),
        "старый: среднее прогноза": float(old_prediction.mean()),
        "новый: среднее прогноза": float(new_prediction.mean()),
        "старый: MAE": old_mae,
        "новый: MAE": new_mae,
        "улучшение MAE": old_mae - new_mae,
        "старый: MAE95": old_mae95,
        "новый: MAE95": new_mae95,
        "улучшение MAE95": old_mae95 - new_mae95,
        "старый: медианная абсолютная ошибка": float(
            old_absolute_error.median()
        ),
        "новый: медианная абсолютная ошибка": float(
            new_absolute_error.median()
        ),
        "старый: P90 абсолютной ошибки": float(
            old_absolute_error.quantile(0.90)
        ),
        "новый: P90 абсолютной ошибки": float(
            new_absolute_error.quantile(0.90)
        ),
        "старый: смещение факт − прогноз": float(
            old_error.mean()
        ),
        "новый: смещение факт − прогноз": float(
            new_error.mean()
        ),
        "старый: дисперсия ошибок": float(
            old_error.var(ddof=1)
        ),
        "новый: дисперсия ошибок": float(
            new_error.var(ddof=1)
        ),
        "старый: СКО ошибок": float(old_error.std(ddof=1)),
        "новый: СКО ошибок": float(new_error.std(ddof=1)),
        "использованный квантиль": float(
            date_df["использованный квантиль"].iloc[0]
        ),
    }


ns_by_date_df = pd.DataFrame([
    build_dynamic_date_row(date_df)
    for _, date_df in common_first_day_df.groupby(
        "score_date",
        sort=True,
    )
])

saldo_first_day_df = first_day_df.loc[
    ~first_day_df["касса закрыта"]
].dropna(subset=[
    "atdtmco_saldo_turn_fact",
    "atdtmco_saldo_turn_pred",
]).copy()


def build_saldo_metric_row(
    evaluation_df: pd.DataFrame,
    variant_name: str,
) -> Dict[str, object]:
    fact = evaluation_df["atdtmco_saldo_turn_fact"]
    prediction = evaluation_df["atdtmco_saldo_turn_pred"]
    error = fact - prediction
    absolute_error = error.abs()
    denominator = float(fact.abs().sum())
    return {
        "вариант": variant_name,
        "количество строк": len(evaluation_df),
        "количество касс": evaluation_df[
            "atdtmco_cashdesk_name"
        ].nunique(),
        "MAE": float(absolute_error.mean()),
        "MAE95": mae95_absolute_fact(
            evaluation_df,
            "atdtmco_saldo_turn_fact",
            "atdtmco_saldo_turn_pred",
        ),
        "WAPE": (
            float(absolute_error.sum()) / denominator
            if denominator > 0
            else np.nan
        ),
        "медианная абсолютная ошибка": float(
            absolute_error.median()
        ),
        "P90 абсолютной ошибки": float(
            absolute_error.quantile(0.90)
        ),
        "смещение факт − прогноз": float(error.mean()),
        "% fallback saldo": float(
            evaluation_df["fallback saldo"].mean()
        ),
    }


saldo_overall_metrics_df = pd.DataFrame([
    build_saldo_metric_row(
        saldo_first_day_df,
        "центральный SARIMA saldo_turn",
    )
])
saldo_by_cashdesk_df = pd.DataFrame([
    dict(
        {"касса": cashdesk_name},
        **build_saldo_metric_row(
            cashdesk_df,
            "центральный SARIMA saldo_turn",
        ),
    )
    for cashdesk_name, cashdesk_df in saldo_first_day_df.groupby(
        "atdtmco_cashdesk_name",
        sort=True,
    )
]).sort_values("MAE", ascending=False).reset_index(drop=True)
saldo_by_date_df = pd.DataFrame([
    dict(
        {"дата скоринга": score_date},
        **build_saldo_metric_row(
            date_df,
            "центральный SARIMA saldo_turn",
        ),
    )
    for score_date, date_df in saldo_first_day_df.groupby(
        "score_date",
        sort=True,
    )
]).sort_values("дата скоринга").reset_index(drop=True)

report_frames = {
    "ns_overall_comparison": ns_overall_comparison_df,
    "ns_by_cashdesk": ns_by_cashdesk_df,
    "ns_cashdesk_summary": ns_cashdesk_summary_df,
    "ns_by_date": ns_by_date_df,
    "saldo_overall_metrics": saldo_overall_metrics_df,
    "saldo_by_cashdesk": saldo_by_cashdesk_df,
    "saldo_by_date": saldo_by_date_df,
    "adaptive_quantile_history": adaptive_update_history_df,
}
for report_name, report_df in report_frames.items():
    report_df.to_csv(
        REPORT_DIR / "{}.csv".format(report_name),
        index=False,
    )
    report_df.to_parquet(
        REPORT_DIR / "{}.parquet".format(report_name),
        index=False,
    )

print(
    "Выбранное окно: {} дней; сила стягивания: {:.0f}".format(
        selected_error_window,
        selected_shrinkage,
    )
)
display(ns_overall_comparison_df.style.format({
    "% невыдач": "{:.2%}",
    "отклонение от цели": "{:+.2%}",
    "среднее факта": "{:,.0f}",
    "среднее прогноза": "{:,.0f}",
    "MAE": "{:,.0f}",
    "MAE95": "{:,.0f}",
    "медианная абсолютная ошибка": "{:,.0f}",
    "P90 абсолютной ошибки": "{:,.0f}",
    "смещение факт − прогноз": "{:,.0f}",
    "дисперсия ошибок": "{:,.0f}",
    "СКО ошибок": "{:,.0f}",
}))
print(
    "Целевой % невыдач: {:.0%}".format(TARGET_BREACH_RATE)
)
display(ns_cashdesk_summary_df.style.format({
    "% касс с улучшением MAE": "{:.2%}",
}))
display(ns_by_cashdesk_df.style.format({
    "среднее факта": "{:,.0f}",
    "старый: % невыдач": "{:.2%}",
    "новый: % невыдач": "{:.2%}",
    "изменение % невыдач": "{:+.2%}",
    "старый: среднее прогноза": "{:,.0f}",
    "новый: среднее прогноза": "{:,.0f}",
    "старый: MAE": "{:,.0f}",
    "новый: MAE": "{:,.0f}",
    "улучшение MAE": "{:+,.0f}",
    "улучшение MAE, %": "{:+.2%}",
    "старый: MAE95": "{:,.0f}",
    "новый: MAE95": "{:,.0f}",
    "улучшение MAE95": "{:+,.0f}",
    "улучшение MAE95, %": "{:+.2%}",
    "старый: медианная ошибка": "{:,.0f}",
    "новый: медианная ошибка": "{:,.0f}",
    "старый: P90 ошибки": "{:,.0f}",
    "новый: P90 ошибки": "{:,.0f}",
    "старый: смещение факт − прогноз": "{:,.0f}",
    "новый: смещение факт − прогноз": "{:,.0f}",
    "старый: дисперсия ошибок": "{:,.0f}",
    "новый: дисперсия ошибок": "{:,.0f}",
    "старый: СКО ошибок": "{:,.0f}",
    "новый: СКО ошибок": "{:,.0f}",
}))
display(ns_by_date_df.style.format({
    "среднее факта": "{:,.0f}",
    "старый: % невыдач": "{:.2%}",
    "новый: % невыдач": "{:.2%}",
    "изменение % невыдач": "{:+.2%}",
    "старый: среднее прогноза": "{:,.0f}",
    "новый: среднее прогноза": "{:,.0f}",
    "старый: MAE": "{:,.0f}",
    "новый: MAE": "{:,.0f}",
    "улучшение MAE": "{:+,.0f}",
    "старый: MAE95": "{:,.0f}",
    "новый: MAE95": "{:,.0f}",
    "улучшение MAE95": "{:+,.0f}",
    "старый: медианная абсолютная ошибка": "{:,.0f}",
    "новый: медианная абсолютная ошибка": "{:,.0f}",
    "старый: P90 абсолютной ошибки": "{:,.0f}",
    "новый: P90 абсолютной ошибки": "{:,.0f}",
    "старый: смещение факт − прогноз": "{:,.0f}",
    "новый: смещение факт − прогноз": "{:,.0f}",
    "старый: дисперсия ошибок": "{:,.0f}",
    "новый: дисперсия ошибок": "{:,.0f}",
    "старый: СКО ошибок": "{:,.0f}",
    "новый: СКО ошибок": "{:,.0f}",
    "использованный квантиль": "{:.3f}",
}))
saldo_metric_format = {
    "MAE": "{:,.0f}",
    "MAE95": "{:,.0f}",
    "WAPE": "{:.2%}",
    "медианная абсолютная ошибка": "{:,.0f}",
    "P90 абсолютной ошибки": "{:,.0f}",
    "смещение факт − прогноз": "{:,.0f}",
    "% fallback saldo": "{:.2%}",
}
display(saldo_overall_metrics_df.style.format(saldo_metric_format))
display(saldo_by_cashdesk_df.style.format(saldo_metric_format))
display(saldo_by_date_df.style.format(saldo_metric_format))


In [ ]:
# РУЧНОЙ ШАГ: переключить на True только после проверки всех таблиц.
WRITE_TO_ORACLE = False

if WRITE_TO_ORACLE:
    oracle_export_df = final_forecasts_df[[
        "score_date",
        "report_date",
        "atdtmco_cashdesk_name",
        "forecast_date",
        "atdtmco_saldo_turn_pred",
        "atdtmco_ns_pred",
        "atdtmco_saldo_turn_fact",
        "atdtmco_ns_fact",
    ]].copy()
    oracle_key_columns = [
        "score_date",
        "atdtmco_cashdesk_name",
        "forecast_date",
    ]
    oracle_numeric_columns = [
        "atdtmco_saldo_turn_pred",
        "atdtmco_ns_pred",
        "atdtmco_saldo_turn_fact",
        "atdtmco_ns_fact",
    ]
    if oracle_export_df.duplicated(oracle_key_columns).any():
        raise ValueError("Перед записью в Oracle найдены дубли")
    if not np.isfinite(
        oracle_export_df[oracle_numeric_columns].to_numpy(dtype=float)
    ).all():
        raise ValueError(
            "Перед записью в Oracle найдены пропуски или невалидные числа"
        )
    oracle_export_df[oracle_numeric_columns] = oracle_export_df[
        oracle_numeric_columns
    ].round(2)
    if engine_cdw is None:
        engine_cdw = await create_cdw_engine()
    oracle.write(
        oracle_export_df,
        engine_cdw,
        ORACLE_TARGET_TABLE,
        batch_size=100_000,
        if_exists="append",
    )
    print("В {} добавлено {:,} строк".format(
        ORACLE_TARGET_TABLE,
        len(oracle_export_df),
    ))
else:
    print("Запись в Oracle отключена: WRITE_TO_ORACLE = False")

## Результаты

Основные файлы сохраняются в `outputs/final_adaptive_retro_schedule`:

- `reports/window_comparison.csv` — лучшая конфигурация для каждого окна;
- `reports/ns_overall_comparison.csv` — итоговое сравнение NS;
- `reports/ns_by_cashdesk.csv` — метрики NS по кассам;
- `reports/ns_cashdesk_summary.csv` — доля касс с улучшением MAE;
- `reports/saldo_overall_metrics.csv` — общие saldo-метрики;
- `reports/saldo_by_cashdesk.csv` — saldo-метрики по кассам;
- `reports/saldo_by_date.csv` — saldo-метрики по датам;
- `reports/adaptive_quantile_history.csv` — последовательное изменение q;
- `final_forecasts.parquet` — все 62 даты × 30 горизонтов.

После выполнения передайте итоговые таблицы для обновления HTML-отчёта. Для повторного запуска без Oracle установите `LOAD_RAW_FROM_CACHE = True`; checkpoints исторических и ретро-прогнозов будут переиспользованы.
